# 📦 Structured Outputs — Extraindo Dados Estruturados com LLMs

Por padrão, um LLM retorna texto livre. Este notebook mostra como **forçar saídas estruturadas** — dicionários, objetos tipados e modelos Pydantic — que podem ser consumidos diretamente por código.

---

## 🗺️ Mapa do Notebook

| Seção | Técnica | Garantia de Estrutura |
|---|---|---|
| **1. String Output** | `StrOutputParser` | Nenhuma — o modelo decide o formato |
| **2. Tool Output** | `@tool` + `ToolOutputParser` | Parcial — o schema da tool guia o modelo |
| **3. Pydantic Output** | `response_format=Model` + `PydanticOutputParser` | Total — validação de tipos e campos |

---

## 🧠 Conceito Central

$$\text{Output Estruturado} = \text{Schema} + \text{Parser} + \text{Validação}$$

Sem estrutura, integrar LLMs em sistemas reais é frágil: qualquer variação de formato pode quebrar o código downstream. **Parsers** criam uma camada de contrato entre o modelo e o resto do sistema.

```
Texto livre:  "Science Fair on Friday with Alice and Bob"
              → Requer regex/NLP para extrair dados

Saída estruturada: {"name": "Science Fair", "date": "Friday", "participants": [...]}
                   → Pronto para salvar em banco, exibir em UI, chamar API
```

> **Por que isso importa em produção:** Agentes reais precisam passar dados entre steps. Um step que extrai uma entidade deve entregar um objeto tipado, não uma string para o próximo step interpretar.

## O que vamos aprender:
- Parsing básico de string com `StrOutputParser`
- Uso de tools para obter saídas estruturadas
- Uso de modelos Pydantic para validação de tipos
- Diferenças entre os tipos de parsers e seus casos de uso

### Setup

In [1]:
from typing import Annotated
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from lib.messages import UserMessage, SystemMessage
from lib.tooling import tool
from lib.llm import LLM
from lib.parsers import (
    StrOutputParser,
    JsonOutputParser, 
    PydanticOutputParser, 
    ToolOutputParser,
)

In [2]:
load_dotenv()

True

In [3]:
chat_model = LLM()

## 1. String Output — A Forma Mais Simples

`StrOutputParser` apenas extrai o campo `.content` da resposta. Não há garantia de estrutura: o modelo pode formatar como quiser.

**Quando usar:** Resumos, explicações, respostas conversacionais — onde o destino é um humano, não código.

**Limitação crítica:** O output abaixo funciona por acaso. Se rodarmos a mesma célula novamente, o modelo pode responder com um formato diferente:

```
Tentativa 1: "Event: Science Fair\nParticipants: Alice and Bob\nDay: Friday"
Tentativa 2: "The event is a Science Fair happening Friday with Alice and Bob."
Tentativa 3: "Name: Science Fair | Date: Friday | People: Alice, Bob"
```

Isso torna `StrOutputParser` inadequado para pipelines que processam o output programaticamente.

In [4]:
messages = [
    SystemMessage(content="Extract the event information."),
    UserMessage(content="Alice and Bob are going to a science fair on Friday.")
]

In [5]:
ai_message = chat_model.invoke(messages)
ai_message

AIMessage(content='Event: Science Fair  \nParticipants: Alice and Bob  \nDay: Friday', role='assistant', tool_calls=None)

In [6]:
parser = StrOutputParser()
print(parser.parse(ai_message))

Event: Science Fair  
Participants: Alice and Bob  
Day: Friday


## 2. Tool Output — Estrutura via Schema de Função

Usar uma `@tool` para extração é um padrão poderoso: o schema JSON da função **instrui o modelo** a preencher campos específicos com tipos específicos.

### Como funciona internamente

```mermaid
flowchart LR
    A["📝 Texto\n'Alice and Bob are going\nto a science fair on Friday'"]
    B["🧠 LLM\ncom tool calendar_event"]
    C["📋 tool_calls\n{name, date, participants}"]
    D["🔧 ToolOutputParser\n.parse(ai_message)"]
    E["✅ dict\n{'name': 'Science Fair',\n'date': 'Friday',\n'participants': ['Alice', 'Bob']}"]

    A --> B --> C --> D --> E
```

### Tool vs. `response_format`: Qual a diferença?

| Característica | Tool (`@tool`) | `response_format` (Pydantic) |
|---|---|---|
| Mecanismo | Model escolhe *quando* chamar | Model é forçado a retornar no schema |
| Flexibilidade | Alta — pode ignorar a tool | Baixa — sempre retorna estruturado |
| Validação | Pelo schema JSON | Pelo Pydantic (tipos Python) |
| Ideal para | Agentes que decidem quando extrair | Pipelines de extração garantida |

> **Dica de design:** Se o objetivo é *sempre* extrair uma entidade, use `response_format`. Se a extração é *opcional* (o agente decide), use uma tool.

In [7]:
@tool
def calendar_event(name:str, date:str, participants:list[str]):
    """Identify name of the event, date when it will happen and all the participants"""
    return {
        "name": name,
        "date": date,
        "participants": participants
    }

In [8]:
messages = [
    SystemMessage(content="Extract the event information."),
    UserMessage(content="Alice and Bob are going to a science fair on Friday.")
]

In [9]:
chat_model_with_tools = LLM(tools=[calendar_event])
ai_message = chat_model_with_tools.invoke(messages)

In [10]:
# This gives us the variables identified by the LLM
parser = ToolOutputParser()
parser.parse(ai_message)[0]["args"]

{'name': 'Science Fair', 'date': 'Friday', 'participants': ['Alice', 'Bob']}

## 3. Pydantic Output — Validação de Tipos Completa

Pydantic adiciona uma camada que tools não oferecem: **validação de tipo em runtime**. Se o LLM retornar `participants` como string em vez de lista, o Pydantic lança um erro antes do dado corrompido chegar ao resto do sistema.

### A Pirâmide de Confiabilidade

```
         ✅ PydanticOutputParser
        ────────────────────────
         Tipos Python validados
         Campos obrigatórios verificados
         Defaults aplicados automaticamente

       ✅ ToolOutputParser / JsonOutputParser
      ──────────────────────────────────────
       Estrutura de dict garantida
       Sem validação de tipo Python

    ⚠️  StrOutputParser
   ──────────────────────────────────────────
    Apenas extrai .content
    Formato livre — pode variar entre chamadas
```

### `Field` com `Annotated` — Por que usar?

```python
# Sem Field: o LLM não tem dica do que colocar se o dado não existir
name: str

# Com Field: você define descrição e default explicitamente
name: Annotated[str, Field(description="Name/Title of the event", default=None)]
```

O `description` dentro de `Field` é incluído no schema JSON enviado ao LLM — funciona como prompt adicional para cada campo.

> **Regra de ouro:** Use `PydanticOutputParser` quando o output vai ser persistido (banco de dados, API externa) ou quando tipos incorretos causariam erros silenciosos no sistema.

In [11]:
class CalendarEvent(BaseModel):
    """A Pydantic model representing a calendar event.
    
    This model defines the structure for calendar events with validation:
    - name: The name/title of the event
    - date: When the event will occur
    - participants: List of people attending the event
    """

    name: Annotated[str, Field(description="Name/Title of the event. Defaults to ''", default=None)]
    date: Annotated[str, Field(description="Date of the event. Defaults to ''", default=None)]
    participants: Annotated[list[str], Field(description="Who will participate. Defaults to ''", default=None)]

In [12]:
messages = [
    SystemMessage(content="Extract the event information."),
    UserMessage(content="Alice and Bob are going to a science fair on Friday.")
]

In [13]:
ai_message = chat_model.invoke(input=messages, response_format=CalendarEvent)
ai_message

AIMessage(content='{"name":"Science Fair","date":"Friday","participants":["Alice","Bob"]}', role='assistant', tool_calls=None)

In [14]:
# This gives us a structured dictionary we can work with programmatically
parser = JsonOutputParser()
parser.parse(ai_message)

{'name': 'Science Fair', 'date': 'Friday', 'participants': ['Alice', 'Bob']}

In [15]:
# This gives us a validated Pydantic model with type checking
parser = PydanticOutputParser(model_class=CalendarEvent)
event:CalendarEvent = parser.parse(ai_message)
event

CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

In [16]:
event.participants

['Alice', 'Bob']